In [1]:
# The pytorch version 2.4 is mostly tested for mamba_ssm Library so Downgrading default Pytorch version
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 54.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 72.9 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 67.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 22.3 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 82.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 MB 3.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
# Based on the GPU requriments of Kaggle and Pytorch version, downloading the appropriate Wheel files
!wget https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.2/causal_conv1d-1.5.2+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

!wget https://github.com/state-spaces/mamba/releases/download/v2.2.5/mamba_ssm-2.2.5+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

--2026-03-29 14:15:39--  https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.2/causal_conv1d-1.5.2+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/726718359/604a2e41-ef67-4125-bb18-e5c1bfafc2be?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-29T15%3A15%3A25Z&rscd=attachment%3B+filename%3Dcausal_conv1d-1.5.2%2Bcu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-29T14%3A15%3A19Z&ske=2026-03-29T15%3A15%3A25Z&sks=b&skv=2018-11-09&sig=uHkzMz7VzMymJDPb2Zmv4KtB7gxrnT7HySJsawjCo8M%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcm

In [3]:
# Installing the Wheel files
!pip install /kaggle/working/causal_conv1d-1.5.2+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
!pip install /kaggle/working/mamba_ssm-2.2.5+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

Processing ./causal_conv1d-1.5.2+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
Processing ./mamba_ssm-2.2.5+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl


In [8]:
import os
import glob
import torch
import numpy as np
import polars as pl
import torch.nn as nn
from mamba_ssm import Mamba
from huggingface_hub import snapshot_download

# --- MODULE 1: DATA INGEST ---
class OpTCDataIngest:
    def __init__(self, repo_id, local_dir="/kaggle/working"):
        self.repo_id = repo_id
        self.local_dir = local_dir

    def download_host_history(self, host_id):
        """Downloads all 10 days for a specific host."""
        for day in range(16, 26):
            snapshot_download(
                repo_id=self.repo_id,
                repo_type="dataset",
                local_dir=self.local_dir,
                allow_patterns=f"sep{day}/*{host_id}.parquet",
                max_workers=4
            )
        print(f"--- Downloaded history for {host_id} ---")

# --- MODULE 2: FEATURE ENGINE ---
class MambaFeatureEngine:
    @staticmethod
    def process_file(file_path, window_size=2048):
        df = pl.read_parquet(file_path)
        # Eager parsing for timezone safety
        ts = df["timestamp"].str.to_datetime(strict=False).dt.convert_time_zone("UTC").dt.replace_time_zone(None)
        
        # Relational & Temporal Features
        df = df.with_columns([
            ts.alias("timestamp"),
            pl.col("action").fill_null("null").hash().mod(100).cast(pl.Float32),
            pl.col("actorID").fill_null("none").hash().mod(1000).cast(pl.Float32),
            pl.col("objectID").fill_null("none").hash().mod(1000).cast(pl.Float32),
            (ts - ts.shift(1)).fill_null(pl.duration(milliseconds=0)).dt.total_milliseconds().cast(pl.Float32).log1p().alias("td_log")
        ]).sort("timestamp")
        
        X = df.select(["action", "actorID", "objectID", "td_log"]).to_numpy()
        y = df.select(pl.col("is_malicious").cast(pl.Int32)).to_numpy()
        
        # Chunking into windows
        X_chunks = [X[i:i+window_size] for i in range(0, len(X)-window_size+1, window_size)]
        y_chunks = [y[i:i+window_size] for i in range(0, len(y)-window_size+1, window_size)]
        
        return torch.tensor(X_chunks, dtype=torch.float32), torch.tensor(y_chunks, dtype=torch.float32)

# --- MODULE 3: THE MODEL ---
class InfiniteHorizonMamba(nn.Module):
    def __init__(self, input_dim=4, d_model=128, d_state=16):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=4, expand=2)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: [Batch, Seq_Len, Features]
        x = self.proj(x)
        x = self.mamba(x)
        return self.head(x)





def run_training_cycle(host_list, repo_id, batch_size=8):
    ingest = OpTCDataIngest(repo_id)
    model = InfiniteHorizonMamba(input_dim=4).to("cuda")
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1000.0]).to("cuda"))

    for host_id in host_list:
        ingest.download_host_history(host_id)
        
        for day in range(16, 26):
            search_pattern = f"/kaggle/working/sep{day}/*{host_id}*.parquet"
            matching_files = glob.glob(search_pattern)
            
            if not matching_files:
                continue
            
            file_path = matching_files[0]
            
            try:
                # 1. Convert to numpy first to fix the UserWarning
                X_np, y_np = MambaFeatureEngine.process_file(file_path)
                if X_np is None: continue

                # 2. Convert to Tensors
                X_full = torch.from_numpy(np.array(X_np)).float()
                y_full = torch.from_numpy(np.array(y_np)).float()

                # 3. SUB-BATCHING LOOP: Process the file in smaller bites
                total_chunks = X_full.size(0)
                day_loss = 0.0
                
                for i in range(0, total_chunks, batch_size):
                    X = X_full[i : i + batch_size].to("cuda")
                    y = y_full[i : i + batch_size].to("cuda")
                    
                    optimizer.zero_grad()
                    logits = model(X)
                    loss = criterion(logits, y)
                    loss.backward()
                    optimizer.step()
                    
                    day_loss += loss.item()
                    
                    # 4. MEMORY CLEANUP: Clear cache to prevent fragmentation
                    del X, y, logits
                    torch.cuda.empty_cache()

                print(f"✅ Host: {host_id} | Day {day} | Avg Loss: {day_loss/(total_chunks/batch_size):.4f}")
                
            except Exception as e:
                print(f"❌ Error on {file_path}: {e}")
                torch.cuda.empty_cache() # Clean up after crash

# --- EXECUTION ---
hosts = ["0618", "0104"] # Add your 100 hosts here
run_training_cycle(hosts, "equalNull/hosts100-labelled-optc")

Fetching 0 files: 0it [00:00, ?it/s]

Fetching 0 files: 0it [00:00, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

--- Downloaded history for 0104 ---
✅ Host: 0104 | Day 16 | Avg Loss: nan
✅ Host: 0104 | Day 17 | Avg Loss: nan
✅ Host: 0104 | Day 18 | Avg Loss: nan
✅ Host: 0104 | Day 19 | Avg Loss: nan
✅ Host: 0104 | Day 20 | Avg Loss: nan
✅ Host: 0104 | Day 21 | Avg Loss: nan
✅ Host: 0104 | Day 22 | Avg Loss: nan
✅ Host: 0104 | Day 23 | Avg Loss: nan
✅ Host: 0104 | Day 24 | Avg Loss: nan
✅ Host: 0104 | Day 25 | Avg Loss: nan
